# M2A4 — Exemplos práticos de aceleração com medições consistentes

Este notebook reorganiza os exemplos da aula para ficar mais rigoroso em termos de medição e mais didático para apresentação.

## O que mudou nesta versão
- **Todos os imports ficam no início**.
- Os exemplos numéricos usam **o mesmo conjunto de vetores aleatórios**, gerado uma única vez.
- Os tempos são medidos em **5 execuções**, para reduzir o efeito de oscilações do sistema.
- Os resultados são **acumulados** em uma estrutura comum para comparação final.
- Ao final, o notebook monta uma **tabela-resumo de tempos e speedup**.

## Objetivo pedagógico
Mostrar que desempenho não depende apenas de “fazer a mesma conta”, mas também de:
- overhead do interpretador;
- vetorização;
- compilação JIT;
- dependência de dados;
- transferência CPU ↔ GPU;
- paralelismo por tarefas independentes.


## Célula 1 — Imports, configuração e infraestrutura de benchmark

Nesta célula preparamos todo o ambiente do notebook.

### O que acontece
1. Importamos as bibliotecas necessárias.
2. Definimos uma semente aleatória para reprodutibilidade.
3. Criamos uma função genérica de benchmark para repetir a mesma operação várias vezes.
4. Criamos a estrutura `benchmark_results`, onde cada experimento vai registrar seus tempos.

### Por que isso é importante
Em demonstrações de desempenho, comparar uma célula isolada com outra pode ser enganoso.  
Ao usar a **mesma infraestrutura de medição**, a comparação fica mais justa e mais fácil de explicar em sala.


In [ ]:
import time
import math
import hashlib
import statistics as stats
import multiprocessing as mp

import numpy as np
import pandas as pd

from numba import njit

try:
    import cupy as cp
    CUPY_AVAILABLE = True
except Exception:
    cp = None
    CUPY_AVAILABLE = False

SEED = 42
REPEATS = 5

rng = np.random.default_rng(SEED)
benchmark_results = []

def benchmark(func, repeats=REPEATS, warmup=False, sync_func=None):
    """Executa uma função várias vezes e retorna lista de tempos e último resultado."""
    if warmup:
        _ = func()
        if sync_func is not None:
            sync_func()

    times = []
    last_result = None
    for _ in range(repeats):
        if sync_func is not None:
            sync_func()
        start = time.perf_counter()
        last_result = func()
        if sync_func is not None:
            sync_func()
        end = time.perf_counter()
        times.append(end - start)
    return times, last_result

def register_result(section, method, times, note=""):
    benchmark_results.append({
        "secao": section,
        "metodo": method,
        "exec_1_s": times[0],
        "exec_2_s": times[1],
        "exec_3_s": times[2],
        "exec_4_s": times[3],
        "exec_5_s": times[4],
        "media_s": float(np.mean(times)),
        "desvio_s": float(np.std(times, ddof=1)) if len(times) > 1 else 0.0,
        "nota": note,
    })

print(f"CuPy disponível? {CUPY_AVAILABLE}")
print(f"Repetições por experimento: {REPEATS}")

## Célula 2 — Dados compartilhados por todos os experimentos numéricos

Agora geramos os vetores que serão reutilizados nos experimentos de:
- loop Python;
- NumPy vetorizado;
- Numba JIT;
- lógica com dependência entre elementos;
- CuPy na GPU.

### O que observar
Ao usar os **mesmos dados de entrada**, evitamos uma comparação injusta causada por vetores diferentes.

### Nota didática
Para simplificar a aula, usamos um caso clássico de HPC:
**multiplicação elemento a elemento de vetores grandes**.  
É um problema excelente para discutir:
- acesso sequencial à memória;
- vetorização;
- compilação JIT;
- aceleração em GPU.


In [ ]:
n = 10_000_000

a = rng.random(n, dtype=np.float64)
b = rng.random(n, dtype=np.float64)

print(f"Tamanho dos vetores: {n:,}")
print(f"Tipo de dado: {a.dtype}")
print(f"Memória aproximada por vetor: {a.nbytes / (1024**2):.2f} MiB")

## Célula 3 — Definição das funções de trabalho

Nesta etapa definimos todas as funções que serão comparadas.

### Experimentos incluídos
1. **Loop Python puro**: a forma mais direta, mas com overhead do interpretador.
2. **NumPy vetorizado**: delega a operação para código nativo otimizado.
3. **Numba JIT**: mantém o estilo de loop, mas compila para código de máquina.
4. **Lógica dependente**: exemplo em que cada saída depende da posição anterior.
5. **Hashing**: exemplo fora do contexto de IA, focado em tarefas independentes.

### Ponto conceitual importante
Nem todo problema é “igualmente vetorizável”.  
Essa distinção será essencial quando compararmos:
- multiplicação simples;
- lógica com dependência entre iterações;
- hashing paralelo.


In [ ]:
def multiply_python_list(a, b):
    return [a[i] * b[i] for i in range(len(a))]

def multiply_numpy(a, b):
    return a * b

@njit
def multiply_numba(a, b):
    c = np.empty(len(a), dtype=np.float64)
    for i in range(len(a)):
        c[i] = a[i] * b[i]
    return c

def logic_python(arr):
    n = len(arr)
    res = np.zeros(n, dtype=np.float64)
    for i in range(1, n):
        if arr[i] > 0.5:
            res[i] = math.sqrt(arr[i]) + res[i - 1]
        else:
            res[i] = arr[i] ** 2 - res[i - 1]
    return res

@njit
def logic_numba(arr):
    n = len(arr)
    res = np.zeros(n, dtype=np.float64)
    for i in range(1, n):
        if arr[i] > 0.5:
            res[i] = math.sqrt(arr[i]) + res[i - 1]
        else:
            res[i] = arr[i] ** 2 - res[i - 1]
    return res

def hash_data(data):
    return hashlib.sha256(data.encode()).hexdigest()

def hash_sequential(inputs):
    return [hash_data(x) for x in inputs]

def hash_parallel(inputs, processes=None):
    with mp.Pool(processes=processes) as pool:
        return pool.map(hash_data, inputs)

## Célula 4 — Loop Python vs NumPy vetorizado vs Numba JIT

Agora comparamos três estratégias para **a mesma operação** sobre os **mesmos vetores**:

- **Python puro**: executa elemento por elemento no interpretador.
- **NumPy**: executa um kernel vetorizado em código nativo.
- **Numba**: compila o loop explícito para código nativo.

### O que observar
- O **resultado numérico** deve ser o mesmo.
- O **tempo** deve cair drasticamente ao sair do loop Python puro.
- Em alguns ambientes, o **Numba pode superar o NumPy**.  
  Isso não significa que “Numba é sempre melhor”, mas que, para um loop simples e muito regular, o código compilado pode ficar extremamente eficiente.

### Atenção metodológica
O Numba será executado com **warm-up** para separar a fase de compilação JIT da fase de execução medida.


In [ ]:
# Warm-up explícito do Numba para compilar antes da medição
_ = multiply_numba(a, b)

times_python, result_python = benchmark(lambda: multiply_python_list(a, b))
times_numpy, result_numpy = benchmark(lambda: multiply_numpy(a, b))
times_numba, result_numba = benchmark(lambda: multiply_numba(a, b))

# Validação numérica
result_python_np = np.asarray(result_python, dtype=np.float64)
print("Python vs NumPy:", np.allclose(result_python_np, result_numpy))
print("NumPy  vs Numba:", np.allclose(result_numpy, result_numba))

register_result("Multiplicação vetorial", "Python loop", times_python, "Loop interpretado com lista Python.")
register_result("Multiplicação vetorial", "NumPy vetor", times_numpy, "Kernel vetorizado em código nativo.")
register_result("Multiplicação vetorial", "Numba JIT", times_numba, "Loop compilado com JIT, sem custo de compilação na medição.")

print("Tempos Python:", [f"{t:.4f}" for t in times_python])
print("Tempos NumPy :", [f"{t:.4f}" for t in times_numpy])
print("Tempos Numba :", [f"{t:.4f}" for t in times_numba])

## Célula 5 — Quando a dependência entre iterações reduz o paralelismo

Nesta célula mudamos a natureza do problema.

### O que muda
Cada posição `res[i]` depende do valor já calculado em `res[i-1]`.

Isso cria uma **dependência de dados** e reduz fortemente a liberdade para:
- vetorização direta;
- paralelismo massivo;
- reorganização agressiva pelo compilador.

### Comparação proposta
- **Python puro**: correto, mas lento.
- **Numba JIT**: mantém a dependência, porém remove o overhead do interpretador.

### Mensagem didática
Aqui o objetivo não é mostrar “mais paralelismo”, e sim mostrar que:
> a estrutura do algoritmo influencia diretamente o quanto ele pode ser acelerado.


In [ ]:
logic_input = a  # reutilizamos o mesmo vetor gerado no início

# Warm-up do Numba nesta função específica
_ = logic_numba(logic_input)

times_logic_python, result_logic_python = benchmark(lambda: logic_python(logic_input))
times_logic_numba, result_logic_numba = benchmark(lambda: logic_numba(logic_input))

print("Logic Python vs Logic Numba:", np.allclose(result_logic_python, result_logic_numba))

register_result("Dependência entre elementos", "Python lógico", times_logic_python, "Cada passo depende do resultado anterior.")
register_result("Dependência entre elementos", "Numba lógico", times_logic_numba, "Mesmo algoritmo, mas compilado para código nativo.")

print("Tempos lógica Python:", [f"{t:.4f}" for t in times_logic_python])
print("Tempos lógica Numba :", [f"{t:.4f}" for t in times_logic_numba])

## Célula 6 — CuPy: mesma multiplicação vetorial, agora na GPU

Aqui mantemos a operação de multiplicação elemento a elemento, mas mudamos o hardware de execução.

### Fluxo completo
1. Dados começam na CPU (host).
2. São transferidos para a GPU (device).
3. A multiplicação é executada na GPU.
4. A GPU é sincronizada para medir corretamente.
5. O resultado pode ser trazido de volta para a CPU.

### Ponto didático central
A GPU pode ser muito rápida na computação, mas o ganho total depende também de:
- custo de transferência CPU ↔ GPU;
- tamanho do problema;
- regularidade do acesso à memória.

### Observação
Se CuPy não estiver disponível na máquina, a célula apenas registra que o experimento foi pulado.


In [ ]:
if CUPY_AVAILABLE:
    a_gpu = cp.asarray(a)
    b_gpu = cp.asarray(b)

    def cupy_mul():
        return a_gpu * b_gpu

    times_cupy, result_cupy = benchmark(
        cupy_mul,
        repeats=REPEATS,
        sync_func=lambda: cp.cuda.Stream.null.synchronize()
    )
    result_cupy_cpu = cp.asnumpy(result_cupy)

    print("NumPy vs CuPy:", np.allclose(result_numpy, result_cupy_cpu))

    register_result("Multiplicação vetorial", "CuPy GPU", times_cupy, "Tempo do kernel na GPU; dados já residentes no device.")
    print("Tempos CuPy  :", [f"{t:.4f}" for t in times_cupy])
else:
    print("CuPy não disponível neste ambiente. Experimento GPU não executado.")

## Célula 7 — Exemplo fora de IA: hashing sequencial vs multiprocessing

Agora saímos do contexto numérico e mostramos um caso de **tarefas independentes**:
calcular SHA-256 para uma coleção de entradas.

### O que este exemplo ensina
- Nem toda aceleração envolve vetores, matrizes ou GPU.
- Alguns problemas escalam bem ao simplesmente dividir as tarefas entre vários processos.
- Esse caso é útil para mostrar **paralelismo por tarefas**, e não apenas por dados.

### Estratégias comparadas
- **Sequencial**: um hash por vez.
- **Multiprocessing**: vários processos calculando hashes em paralelo.

### Nota prática
A criação de processos tem overhead.  
Por isso, o tamanho da carga de trabalho precisa ser grande o suficiente para justificar o paralelismo.


In [ ]:
hash_inputs = [f"dado_{i}" for i in range(100_000)]

times_hash_seq, result_hash_seq = benchmark(lambda: hash_sequential(hash_inputs))
times_hash_mp, result_hash_mp = benchmark(lambda: hash_parallel(hash_inputs))

print("Hash sequencial vs multiprocessing:", result_hash_seq == result_hash_mp)

register_result("Hashing", "Sequencial", times_hash_seq, "Uma tarefa por vez, sem paralelismo de processos.")
register_result("Hashing", "Multiprocessing", times_hash_mp, "Distribuição de tarefas independentes entre múltiplos processos.")

print("Tempos hash seq:", [f"{t:.4f}" for t in times_hash_seq])
print("Tempos hash mp :", [f"{t:.4f}" for t in times_hash_mp])

## Célula 8 — Tabela-resumo com médias, desvio e speedup

Nesta última etapa consolidamos todos os resultados em uma tabela única.

### Métricas apresentadas
- tempos individuais das 5 execuções;
- média;
- desvio padrão;
- **speedup relativo ao baseline da mesma seção**.

### Como interpretar
O speedup é calculado sempre contra o método de referência da própria seção:
- em **multiplicação vetorial**, o baseline é o **Python loop**;
- em **dependência entre elementos**, o baseline é o **Python lógico**;
- em **hashing**, o baseline é o **sequencial**.

Isso evita comparações sem sentido entre problemas diferentes.


In [ ]:
df = pd.DataFrame(benchmark_results)

baseline_map = {}
for secao in df["secao"].unique():
    sec_df = df[df["secao"] == secao]
    baseline_method = sec_df.iloc[0]["metodo"]
    baseline_time = sec_df.iloc[0]["media_s"]
    baseline_map[secao] = (baseline_method, baseline_time)

def compute_speedup(row):
    _, baseline_time = baseline_map[row["secao"]]
    return baseline_time / row["media_s"]

df["speedup_vs_baseline"] = df.apply(compute_speedup, axis=1)

df_display = df.copy()
for col in ["exec_1_s", "exec_2_s", "exec_3_s", "exec_4_s", "exec_5_s", "media_s", "desvio_s", "speedup_vs_baseline"]:
    df_display[col] = df_display[col].map(lambda x: round(float(x), 4))

display(df_display.sort_values(["secao", "media_s"], ascending=[True, True]))

## Célula 9 — Leitura final dos resultados

Use a tabela final para fechar a aula com quatro mensagens-chave:

1. **Python puro** é simples, mas paga caro pelo overhead do interpretador.
2. **NumPy** acelera muito bem problemas vetorizáveis e regulares.
3. **Numba** pode alcançar ou até superar o NumPy em loops simples, desde que o padrão de acesso seja favorável.
4. **GPU** não é “mágica”: o ganho depende de tamanho do problema, custo de transferência e regularidade da computação.
5. **Multiprocessing** mostra que paralelismo também vale para tarefas não relacionadas a IA, como hashing.

### Sugestão de discussão em sala
Pergunte aos alunos:
- Em qual caso o hardware foi mais importante?
- Em qual caso o algoritmo foi mais importante?
- Em qual caso o overhead anulou parte do ganho?
